# 03. pandas 기본

* CSV 파일을 데이터프레임으로 불러와 **확인 → 통계 → 정렬 → 선택 → 가공 → 결측치 → 시각화** 흐름을 익힙니다.
* 앞서 배운 **Series·DataFrame** 개념이 실제 데이터에서 어떻게 쓰이는지 확인합니다.

> 📁 이 노트북은 **`data/scores.csv`** 파일을 사용합니다.
> 노트북과 같은 위치에 **`data`** 폴더를 만들고 그 안에 `scores.csv`를 넣어주세요.

In [ ]:
import pandas as pd

## CSV 파일로 데이터프레임 생성
* **`pd.read_csv(파일경로)`**
    * 인덱스 라벨 : 기본은 0,1,2 … (RangeIndex)
    * 컬럼 라벨 : CSV의 첫 번째 행
    * 컬럼 자료형 : 자동 추론

In [ ]:
# ----------------------------------------------------------
# 그냥 읽기 (학번이 정수로 인식됨)
# ----------------------------------------------------------

pd.read_csv('data/scores.csv')

In [ ]:
# ----------------------------------------------------------
# 옵션을 주어 읽기
#   - dtype     : 컬럼의 자료형 지정 (학번을 문자열로)
#   - index_col : 인덱스 라벨로 사용할 컬럼 지정
# ----------------------------------------------------------

df = pd.read_csv('data/scores.csv',
                 dtype={'학번': 'str'},
                 index_col='학번')
df

## 데이터프레임 확인

#### 미리보기
* `df.head(n)` : 상위 n개 (기본 5)
* `df.tail(n)` : 하위 n개 (기본 5)
* `df.sample(n)` : 랜덤 n개 (기본 1)

In [ ]:
df.head(3)

In [ ]:
df.tail(3)

In [ ]:
df.sample(3)

#### 정보 확인 : `df.info()`
* 인덱스 종류·범위, 컬럼 개수, **Null 여부**, 컬럼별 자료형, 메모리 사용량을 한눈에

In [ ]:
df.info()

#### 주요 속성
* `df.shape` : (행, 열) 튜플
* `df.columns` : 컬럼 라벨
* `df.index` : 인덱스 라벨
* `df.dtypes` : 컬럼별 자료형

In [ ]:
print('shape  :', df.shape)
print('columns:', list(df.columns))
print('index  :', list(df.index))
print()
print(df.dtypes)

> 💡 **참고** — 가끔 쓰는 속성
> * `df.values` : 값 전체를 넘파이 배열로
> * `df.size` : 전체 원소 수 (행×열)
> * `df.T` : 전치(행↔열 바꾸기)

In [ ]:
# 참고용 : 필요할 때만 사용
print('size :', df.size)   # 행*열
df.T                        # 전치 (행과 열을 바꿔서 보기)

## 데이터 통계
* 전체 요약통계 : `df.describe()` (숫자형 컬럼만, NaN은 무시)
* 컬럼별 통계 : `df['컬럼'].mean()`, `.min()`, `.max()`, `.median()`, `.std()` …

In [ ]:
df.describe()

In [ ]:
# 국어 점수의 평균 (NaN은 무시하고 계산)
df['국어'].mean()

## 빈도수
* 범주형 데이터의 값별 개수 : `df['컬럼'].value_counts()`

In [ ]:
# 성별 빈도수
df['성별'].value_counts()

## 정렬
* `df.sort_values(by='컬럼')` : 값으로 정렬
* `df.sort_index()` : 인덱스로 정렬
* 옵션 `ascending=False` : 내림차순

In [ ]:
# 국어 점수 오름차순
df.sort_values(by='국어')

In [ ]:
# 국어 점수 내림차순
df.sort_values(by='국어', ascending=False)

## 데이터 선택

### 컬럼 선택
* `df['컬럼']` → **Series**
* `df[['컬럼']]` → **DataFrame**
* `df[['컬럼1','컬럼2']]` → 여러 컬럼

In [ ]:
# '이름' 컬럼 → Series
df['이름']

In [ ]:
# '이름','국어' 컬럼 → DataFrame
df[['이름', '국어']]

### 라벨로 선택 : `df.loc[행, 열]`
* 행·열을 **라벨**로 지정 (라벨 슬라이스는 끝 포함)

In [ ]:
# 학번 0001 학생의 전체 데이터
df.loc['0001']

In [ ]:
# 학번 0001~0003 학생의 국어~수학 점수
df.loc['0001':'0003', '국어':'수학']

In [ ]:
# 학번 0001, 0005 학생의 이름, 수학
df.loc[['0001', '0005'], ['이름', '수학']]

### 정수 위치로 선택 : `df.iloc[행, 열]`
* 행·열을 **정수 위치**로 지정 (슬라이스는 끝 미포함, 파이썬과 동일)

In [ ]:
# 0번째 행
df.iloc[0]

In [ ]:
# 0~2번째 행의 마지막 열
df.iloc[0:3, -1]

### 조건으로 선택 : `df.loc[조건]`
* 비교연산 : `>`, `<`, `>=`, `<=`, `==`, `!=`
* 논리연산 : `&`(and), `|`(or), `~`(not) — 각 조건은 **소괄호로** 묶기
* 멤버연산 : `.isin([값들])`
* 결측치 : `.isnull()`, `.notnull()`

In [ ]:
# 국어 점수가 100점인 학생
cond = df['국어'] == 100
df.loc[cond]

In [ ]:
# 국어·영어·수학 모두 100점인 학생
cond1 = df['국어'] == 100
cond2 = df['영어'] == 100
cond3 = df['수학'] == 100

df.loc[cond1 & cond2 & cond3]

In [ ]:
# 국어 점수가 null인 학생
cond = df['국어'].isnull()
df.loc[cond]

## 열 추가 / 삭제

#### 파생컬럼 추가
* **`df['새컬럼'] = 데이터`** (컬럼이 없으면 추가, 있으면 수정)
* 컬럼 간 연산에서 **NaN이 있으면 결과도 NaN** (전파)
* 단, `mean()` 같은 집계함수는 NaN을 무시하고 계산

In [ ]:
# 국어~수학의 평균을 '평균' 컬럼으로 추가
df['평균'] = df.loc[:, '국어':'수학'].mean(axis=1).round(2)
df

#### 열 삭제
* `df.drop('컬럼', axis=1)`

In [ ]:
# 평균 컬럼 삭제
df = df.drop('평균', axis=1)
df

## 결측치 처리

#### 결측치 확인
* 컬럼별 결측치 개수 : `df.isnull().sum()`

In [ ]:
df.isnull().sum()

#### 결측치 삭제
* `df.dropna()` : 결측치가 있는 **행 전체** 삭제
* `df.dropna(subset=['컬럼'])` : 특정 컬럼에 결측이 있는 행만 삭제

In [ ]:
# 결측치가 있는 행 삭제 (결과만 확인 → 원본 df는 그대로)
df.dropna()

#### 이웃 값으로 채우기
* `df.ffill()` : 이전(위) 값으로 채우기
* `df.bfill()` : 다음(아래) 값으로 채우기

In [ ]:
# 이전 값으로 채우기 (원본은 그대로)
df.ffill()

#### 컬럼별로 다른 값 채우기
* `df.fillna({'컬럼1':값1, '컬럼2':값2})`
* 예: 국어는 평균으로, 수학은 0으로

In [ ]:
# 국어는 평균, 수학은 0으로 채우기 (원본은 그대로)
df.fillna({'국어': df['국어'].mean(), '수학': 0})

#### 결측치를 0으로 채우기 (원본에 반영)
* 재할당으로 **원본 df에 반영**합니다. (뒤의 astype(int)를 위해 결측치를 없앰)

In [ ]:
# 최종: 0으로 채워 원본에 반영
df = df.fillna(0)
df

## 자료형 변경
* `df.astype({'컬럼': '자료형'})`
* `df['컬럼'].astype('자료형')`

In [ ]:
# 국어, 수학 컬럼을 정수형으로 변경
df = df.astype({'국어': 'int', '수학': 'int'})
df.dtypes

## 시각화
* `df.plot()`
    * 기본 그래프 : 선그래프 / x축 : 인덱스 / y축 : 숫자형 컬럼
* 주요 옵션 : `kind`(종류), `x`, `y`, `figsize`(크기), `title`(제목), `color`(색)

#### 한글 폰트 설정
* matplotlib의 **기본 폰트에는 한글 글자 모양이 없어서**, 한글을 그리면 네모(□)로 깨집니다.
* 그래서 그래프를 그리기 전에 **한글이 들어있는 폰트로 기본값을 바꿔줘야** 합니다.
* 폰트는 운영체제마다 이름이 달라서(윈도우 `Malgun Gothic`, 맥 `AppleGothic`, 리눅스·코랩 `NanumGothic`),
  아래 셀은 **실행 중인 OS를 자동으로 감지**해 알맞은 폰트를 지정합니다.

> 💡 **`plt.rc('그룹', 키=값)`** 은 matplotlib의 **전역 기본 설정**(`rcParams`)을 바꾸는 함수입니다.
> (`rc` = runtime configuration, 실행 환경 설정)
> * `plt.rc('font', family='...')` → 앞으로 그리는 **모든 그래프의 기본 글꼴**을 지정 (한 번만 실행하면 계속 적용)
> * `plt.rc('axes', unicode_minus=False)` → **음수 기호(−)** 를 일반 하이픈(-)으로 처리.
>   한글 폰트엔 유니코드 마이너스 글자가 없는 경우가 많아, 이 줄이 없으면 음수가 네모로 깨집니다.
>   그래서 한글 폰트 설정과 거의 **항상 세트**로 씁니다.

In [ ]:
# ------------------------------------------------
# 한글 폰트 자동 설정 (환경에 맞게 알아서 선택)
# ------------------------------------------------
import platform
import matplotlib.pyplot as plt

system = platform.system()   # 실행 중인 OS 이름 반환
if system == 'Windows':
    plt.rc('font', family='Malgun Gothic')      # 윈도우
elif system == 'Darwin':                        # Darwin = macOS(맥)
    plt.rc('font', family='AppleGothic')        # 맥
else:
    plt.rc('font', family='NanumGothic')        # 리눅스·코랩
    # 코랩에서 폰트가 없다면 아래 한 줄을 먼저 실행하세요
    # !apt -qq install fonts-nanum && rm -rf ~/.cache/matplotlib

plt.rc('axes', unicode_minus=False)             # 음수 기호(-) 깨짐 방지

In [ ]:
# ------------------------------
# 국어 점수 막대그래프
#  - kind='bar', x=이름, y=국어
# ------------------------------
df.plot(kind='bar', x='이름', y='국어',
        figsize=(6, 3), color='#D62728',
        title='국어 점수')
plt.show()

In [ ]:
# ------------------------------
# 학생별 국어·영어·수학 막대그래프
# ------------------------------
df.plot(kind='bar', x='이름',
        y=['국어', '영어', '수학'],
        figsize=(12, 5),
        color=['#D62728', '#FFD700', '#17BECF'],
        title='학생별 점수')
plt.show()